[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Simone-Alghisi/HMD-Lab/blob/master/notebooks/4_nlg.ipynb)

On Colab:
1. Switch to a GPU Runtime by clicking on *Runtime > Change runtime type > T4 GPU*
2. Run the cell below

In [ ]:
# For Google Colab only
!git clone https://github.com/Simone-Alghisi/HMD-Lab.git
%cd /content/HMD-Lab/notebooks 

![system_architecture](./assets/system.jpg)

# Natural Language Generation (NLG)

Natural Language Generation (NLG) is the component that converts compact, machine-readable actions (produced by the [Dialogue Manager](./3_dm.ipynb)) into natural language sentences. In task-oriented dialogue systems the NLG is responsible for lexicalizing DM actions (along with the dialogue state) and producing polite, context-appropriate answers.

Typical NLG responsibilities:

- Produce a response in natural language for DM actions, such as `request_info(pizza_size)` or `confirmation(pizza_ordering)`.
- Respect tone and politeness constraints (short, friendly, concise responses for chatbots).

Design tips:
- Provide the NLG with a clear action and the minimal context needed (intent, slots, any user-provided values).
- Encourage concise replies and specify the style (polite, friendly, short).
- For deterministic behavior, prefer templates or add a few-shot example showing the exact phrasing you want.


## Question

You want to design an OrderBot to collect the user's pizza order.

Consider the following system action: `request_info(pizza_size)`

1. What should be the output of the NLG?

2. What if the action is `confirmation(pizza_ordering)`

3. How would you design a system to answer the user?


### Solution

1. Many possible answers:
   - `"What size would you like for your pizza?"`
   - `"Great! What about the size?"`
   - `"So you would like a margherita pizza. What size? We have small, medium, and large"`
  
2. We will produce a summary of the order and ask for a confirmation (Do we need anything else?) 

3. Depends on different factors, including resources, latency, and domain:
   - Template-based (e.g., `"Would you like a {} pizza?"`)
   - Reponse selection (among a set of candidates)
   - Response generation

## Code

### Template

In [1]:
# Template-based NLG example

def template_nlg(action: str, state: dict) -> str:
    """Produce a deterministic utterance using simple templates."""
    # helper to safely read slot
    def s(slot_name, default=""):
        return state.get("slots", {}).get(slot_name) or default

    if action.startswith("request_info("):
        slot = action[len("request_info("):-1]
        if slot == "pizza_size":
            pizza_type = s("pizza_type", "your pizza")
            sentence = f"Which size would you like for the {pizza_type} ?"
        elif slot == "pizza_count":
            sentence = "How many pizzas would you like?"
        elif slot == "pizza_type":
            sentence = "Which pizza would you like?"
        return sentence

    if action.startswith("confirmation("):
        intent = action[len("confirmation("):-1]
        if intent == "pizza_ordering":
            slots = state.get("slots", {})
            size = slots.get("pizza_size") or "(size not provided)"
            ptype = slots.get("pizza_type") or "(type not provided)"
            count = slots.get("pizza_count") or 1
            return f"Just to confirm: {count} {size} {ptype}. Is that correct?"

    if action == "provide_capabilities":
        return "I am a bot designed to help you order pizzas. Would you like to order one now?"

    # fallback
    return "Sorry, I didn't understand. Could you rephrase?"

In [2]:
template_nlg("provide_capabilities", {})

'I am a bot designed to help you order pizzas. Would you like to order one now?'

In [3]:
action = "request_info(pizza_type)"

template_nlg(action, {})

'Which pizza would you like?'

In [4]:
state = {
    "intent": "pizza_ordering",
    "slots": {"pizza_type": "margherita", "pizza_size": "medium", "pizza_count": 1}
}

template_nlg("confirmation(pizza_ordering)", state)

'Just to confirm: 1 medium margherita. Is that correct?'

Templates-based solutions are fast and reliable. However
1. Lines of codes increase based on the number of states
2. Responses are fixed

How do we solve this?

### Generation w. LLMs

In [5]:
import sys

sys.path.append("..")

from utils import MODELS
from transformers import AutoTokenizer

model_name, InitModel, prepare_text = MODELS["qwen3"]

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = InitModel(
    model_name,
    dtype="auto",
    device_map="cuda:0",
)

/home/simone/miniconda3/envs/hmd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  1.93it/s]


In [6]:
import torch

from notebooks.notebook_utils import display_conversation
from models.qwen3 import prepare_text

task_prompt = """You are given the Next Best Action (NBA) and the Dialogue State (DS).
The Next Best Action is a compact, machine-readable representation of what the dialogue manager wants to do next in the conversation.
The Dialogue State contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}

Based on the Next Best Action and the Dialogue State, generate a natural language response that is polite, concise, and contextually appropriate.
Output at most 50 words.
"""

In [7]:
nlg_input = """NBA: request_info(pizza_size)
DS: {
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": null,
    "pizza_type": "margherita",
    "pizza_count": null
  }
}"""

messages = [
    {
        "role": "system", 
        "content": task_prompt
    }
]

text = prepare_text(nlg_input, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    # max_new_tokens limits the length of the generated response
    generated_ids = model.generate(**model_inputs, max_new_tokens=50).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, nlg_input, content)


### Conversation

**System:** You are given the Next Best Action (NBA) and the Dialogue State (DS).
The Next Best Action is a compact, machine-readable representation of what the dialogue manager wants to do next in the conversation.
The Dialogue State contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}

Based on the Next Best Action and the Dialogue State, generate a natural language response that is polite, concise, and contextually appropriate.
Output at most 50 words.


**User:** NBA: request_info(pizza_size)
DS: {
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": null,
    "pizza_type": "margherita",
    "pizza_count": null
  }
}

**Assistant:** Could you please let me know the size of the pizza you'd like? We have options like small, medium, large, or extra large. Your favorite is margherita!

In [8]:
nlg_input = """NBA: confirmation(pizza_ordering)
DS: {
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": "medium",
    "pizza_type": "margherita",
    "pizza_count": 2
  }
}"""

messages = [
    {
        "role": "system", 
        "content": task_prompt
    }
]

text = prepare_text(nlg_input, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    # max_new_tokens limits the length of the generated response
    generated_ids = model.generate(**model_inputs, max_new_tokens=50).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, nlg_input, content)


### Conversation

**System:** You are given the Next Best Action (NBA) and the Dialogue State (DS).
The Next Best Action is a compact, machine-readable representation of what the dialogue manager wants to do next in the conversation.
The Dialogue State contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}

Based on the Next Best Action and the Dialogue State, generate a natural language response that is polite, concise, and contextually appropriate.
Output at most 50 words.


**User:** NBA: confirmation(pizza_ordering)
DS: {
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": "medium",
    "pizza_type": "margherita",
    "pizza_count": 2
  }
}

**Assistant:** Great! You'd like 2 medium margherita pizzas. Your order is confirmed. Enjoy your meal!

Using generative models (such as LLMs) has several advantages w.r.t. a template-based approach:
- responses are more varied and may engage the user more
- we don't have to write one sentence for each state, possibly reducing the number of lines of code

However, we do not have the guarantees of templates or response selection. Adding to the template `"Plase be polite"` does not mean that our model will not swear against the customer (although less likely if you use a well-known LLM).

## Evaluation

Question: *How can we evalute the NLG component?*

Think about:
1. What is the input
2. What is the expected output

### Example
 
Given the current NBA:
`NBA: request_info(pizza_size)`
```json
{
    "intent": "pizza_ordering", 
    "slots": {
        "pizza_type": "margherita",
        "pizza_size": null,
        "pizza_count": null
    }
}
```

what would you do next?

#### Solution

Generate the answer! But...

1. There are many possible answers:
   - `"What size would you like for your pizza?"`
   - `"Great! What about the size?"`
   - `"So you would like a margherita pizza. What size? We have small, medium, and large"`

2. How do we compare the ground-truth answer with the prediction?

#### Automatic Metrics
- BLEU
- F1-Score
- BertScore

A good idea to get a rough understanding, but they do not correlate with human judgment

#### Human Evaluation
Provide to a (pool of) human judge(s):
- the dialogue history 
- the NLG response

and let them annotate the response.

*Not as easy as it seems*. We need to:
- specify the metrics that we want to examine
  - grammatical correctness
  - coherence with the dialogue history
  - appropriateness
  - ...
- provide a set of guidelines for the annotation
- recruit and PAY the annotators
- verify the agreement among the annotators 